# Урок 2. Эффективность алгоритмов

11 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 1](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-01.ipynb) · [Урок 3 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-03.ipynb)

---

Асимптотика на практике. Ограничения по времени и памяти. Как оценить, уложится ли решение. Оптимизация: замена структуры данных.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "11А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-02", name=ФИО, klass=Класс)

## Разбираемся

### Почему одна программа быстрая, а другая нет

Обе программы правильные, обе дают верный ответ. Но одна считает
мгновенно, а вторая думает полчаса. Разница не в языке и не в железе —
в **алгоритме**.

Чтобы говорить об этом строго, нужен способ измерять эффективность
независимо от компьютера. Такой способ есть: считать не секунды,
а **количество операций** как функцию размера входных данных.

### O-нотация

Записывают так: если для входа размера n алгоритм выполняет примерно
$n^2$ операций, говорят, что его сложность $O(n^2)$.

Константы и младшие слагаемые отбрасывают: $3n^2 + 5n + 100$ — это
всё равно $O(n^2)$, потому что при больших n именно квадрат решает всё.

| Сложность | Название | 1000 элементов | 1 000 000 элементов |
|---|---|---|---|
| $O(1)$ | константная | мгновенно | мгновенно |
| $O(\log n)$ | логарифмическая | 10 шагов | 20 шагов |
| $O(n)$ | линейная | 1 тыс. | 1 млн |
| $O(n \log n)$ | линейно-логарифмическая | 10 тыс. | 20 млн |
| $O(n^2)$ | квадратичная | 1 млн | 10¹² — часы |
| $O(2^n)$ | экспоненциальная | невозможно | невозможно |

Обратите внимание на последнюю строку: экспоненциальный алгоритм
бесполезен уже при n = 50, сколько бы у вас ни было компьютеров.

### Как определить сложность по коду

Правило простое: **считайте вложенность циклов** по размеру данных.

```python
for x in массив:            # O(n)
    ...

for x in массив:            # O(n²)
    for y in массив:
        ...

while n > 0:                # O(log n)
    n //= 2
```

Последний случай важен: если на каждом шаге величина делится пополам,
шагов будет около $\log_2 n$. Именно так работает двоичный поиск.

### Скрытая сложность встроенных операций

Частая ошибка — не учитывать стоимость операций, которые выглядят
как одна строка.

| Операция | Сложность |
|---|---|
| `список[i]` | $O(1)$ |
| `элемент in список` | $O(n)$ |
| `элемент in множество` | $O(1)$ |
| `элемент in словарь` | $O(1)$ |
| `список.append(x)` | $O(1)$ |
| `список.insert(0, x)` | $O(n)$ |
| `sorted(список)` | $O(n \log n)$ |
| `сумма += строка` в цикле | $O(n^2)$ |

Две строки этой таблицы решают судьбу многих программ.

**Проверка вхождения.** `x in список` перебирает все элементы,
а `x in множество` использует хеш-таблицу и работает за константу.
Если внутри цикла по n элементам проверять вхождение в список
из n элементов, получится $O(n^2)$ вместо $O(n)$.

**Склейка строк.** Каждое `строка += кусок` создаёт **новую** строку
и копирует всё содержимое. Тысяча склеек — миллион операций копирования.
Правильно копить куски в списке и один раз собрать через `join`.

### Как ускорить программу

Порядок действий при оптимизации:

1. **Убедиться, что программа вообще медленная.** Измерьте время.
2. **Найти узкое место.** Обычно это один цикл, а не весь код.
3. **Сменить структуру данных.** Список на множество или словарь —
   самое частое и самое эффективное решение.
4. **Убрать лишние проходы.** Если по данным можно пройти один раз
   вместо трёх — сделайте это.
5. **Сменить алгоритм.** Если сложность квадратичная, никакая
   микрооптимизация не спасёт.

> Правило: **сначала правильно, потом быстро.** Оптимизировать
> неверную программу бессмысленно.

## Смотрим, как это работает

### Пример 1. Замеряем время

In [ ]:
import time

def замер(функция, *аргументы):
    начало = time.perf_counter()
    результат = функция(*аргументы)
    прошло = time.perf_counter() - начало
    return результат, прошло


def сумма_циклом(n):
    итог = 0
    for i in range(1, n + 1):
        итог += i
    return итог


def сумма_формулой(n):
    return n * (n + 1) // 2


for n in [100000, 1000000]:
    _, t1 = замер(сумма_циклом, n)
    _, t2 = замер(сумма_формулой, n)
    print(f"n = {n:>9}: цикл {t1 * 1000:>8.2f} мс, формула {t2 * 1000:>8.4f} мс")

Цикл имеет сложность $O(n)$ и растёт вместе с n. Формула — это $O(1)$,
её время не зависит от размера входа вовсе.

Вывод не в том, что надо всегда искать формулу, а в том, что смена
сложности даёт выигрыш, недостижимый никакими другими способами.

### Пример 2. Список против множества

Самая полезная оптимизация в заданиях ЕГЭ.

In [ ]:
данные = list(range(20000))
искомые = list(range(0, 20000, 7))

# Медленно: поиск в списке
def через_список(данные, искомые):
    найдено = 0
    for x in искомые:
        if x in данные:
            найдено += 1
    return найдено


# Быстро: поиск в множестве
def через_множество(данные, искомые):
    набор = set(данные)
    найдено = 0
    for x in искомые:
        if x in набор:
            найдено += 1
    return найдено


_, t1 = замер(через_список, данные, искомые)
_, t2 = замер(через_множество, данные, искомые)

print(f"Через список:    {t1 * 1000:>8.1f} мс")
print(f"Через множество: {t2 * 1000:>8.1f} мс")
print(f"Ускорение в {t1 / t2:.0f} раз")

Одна строка `набор = set(данные)` дала ускорение в десятки раз.
Логика та же, результат тот же — изменилась только структура данных.

Это первое, что нужно проверять, когда программа на ЕГЭ не укладывается
во время.

### Пример 3. Склейка строк

In [ ]:
def склейка_плюсом(n):
    результат = ""
    for i in range(n):
        результат += "x"
    return len(результат)


def склейка_через_join(n):
    куски = []
    for i in range(n):
        куски.append("x")
    return len("".join(куски))


for n in [50000, 200000]:
    _, t1 = замер(склейка_плюсом, n)
    _, t2 = замер(склейка_через_join, n)
    print(f"n = {n:>7}: через += {t1 * 1000:>7.1f} мс, через join {t2 * 1000:>7.1f} мс")

Разница видна и растёт с размером: при увеличении n вдвое время
склейки через `+=` растёт вчетверо — характерный признак
квадратичной сложности.

Строго говоря, современный Python частично оптимизирует такую склейку,
поэтому разрыв меньше теоретического. Но полагаться на эту оптимизацию
не стоит: она работает не всегда.

## Пробуем сами

### Задача 1. Определите сложность

По описанию алгоритма верните его сложность строкой:
`"O(1)"`, `"O(log n)"`, `"O(n)"`, `"O(n log n)"` или `"O(n^2)"`.

| Алгоритм | Сложность |
|---|---|
| `"взять элемент по индексу"` | O(1) |
| `"найти максимум перебором"` | O(n) |
| `"двоичный поиск"` | O(log n) |
| `"сортировка встроенной функцией"` | O(n log n) |
| `"сравнить все пары элементов"` | O(n^2) |

In [ ]:
def сложность(алгоритм):
    return ...

In [ ]:
si.check("1", сложность, [
    ("взять элемент по индексу", "O(1)"),
    ("найти максимум перебором", "O(n)"),
    ("двоичный поиск", "O(log n)"),
    ("сортировка встроенной функцией", "O(n log n)"),
    ("сравнить все пары элементов", "O(n^2)"),
])

### Задача 2. Ускорьте поиск пересечения

Даны два списка. Верните **отсортированный** список элементов,
входящих в оба.

Наивное решение с двумя вложенными циклами работает за $O(n^2)$.
Сделайте за $O(n)$, воспользовавшись множествами.

In [ ]:
def пересечение(первый, второй):
    return ...

In [ ]:
si.check("2", пересечение, [
    (([1, 2, 3, 4], [3, 4, 5]), [3, 4]),
    (([1, 2], [3, 4]), []),
    (([1, 1, 2], [2, 2, 1]), [1, 2]),
])

### Задача 3. Количество шагов двоичного поиска

Сколько шагов сделает двоичный поиск в отсортированном массиве
из `n` элементов в худшем случае?

На каждом шаге область поиска делится пополам. Реализуйте подсчёт
циклом, а не формулой.

In [ ]:
def шагов_поиска(n):
    return ...

In [ ]:
si.check("3", шагов_поиска, [
    (1, 1),
    (2, 2),
    (1000, 10),
    (1000000, 20),
    (0, 0),
])

## Домашнее задание

### Домашнее задание 1. Быстрый подсчёт дубликатов

Верните количество элементов, которые встречаются в списке
**более одного раза**.

Решите за один проход с помощью словаря, а не вложенными циклами.

In [ ]:
def дубликатов(элементы):
    return ...

In [ ]:
si.check("дз1", дубликатов, [
    ([1, 2, 2, 3, 3, 3], 2),
    ([1, 2, 3], 0),
    ([], 0),
    ([5, 5, 5, 5], 1),
])

### Домашнее задание 2. Пара с заданной суммой

Есть ли в списке два **разных по позиции** элемента, дающих в сумме
заданное число? Верните `True` или `False`.

Наивное решение перебирает все пары за $O(n^2)$. Сделайте за $O(n)$:
идите по списку и для каждого элемента проверяйте, встречалось ли
уже число, дополняющее его до нужной суммы.

In [ ]:
def есть_пара(числа, сумма):
    return ...

In [ ]:
si.check("дз2", есть_пара, [
    (([1, 2, 3, 9], 5), True),
    (([1, 2, 3], 10), False),
    (([5, 5], 10), True),
    (([5], 10), False),
    (([], 0), False),
])

### Домашнее задание 3. Оценка числа операций

По сложности алгоритма и размеру входа верните примерное количество
операций.

Для `"O(log n)"` считайте округление вниз от логарифма по основанию 2,
для `"O(n log n)"` — произведение n на этот логарифм.

In [ ]:
def операций(сложность, n):
    return ...

In [ ]:
si.check("дз3", операций, [
    (("O(1)", 1000), 1),
    (("O(log n)", 1024), 10),
    (("O(n)", 1000), 1000),
    (("O(n log n)", 1024), 10240),
    (("O(n^2)", 1000), 1000000),
])

---

### Полезное правило для экзамена

Если в задании сказано «в файле 100 000 чисел», прикиньте заранее:

* $O(n)$ — 100 тысяч операций, мгновенно;
* $O(n \log n)$ — 1,7 миллиона, доли секунды;
* $O(n^2)$ — 10 миллиардов, **не дождётесь**.

Значит, вложенных циклов по всем данным быть не должно. Это правило
сразу отсекает неверный подход — ещё до того, как вы напишете
первую строку.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 1](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-01.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 3 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-03.ipynb)